# 🎬 Fast Wan 2.1 Video Generation on Google Colab (Tesla T4 16GB)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/wan2.1-t4-colab/blob/main/wan2_1_t4_colab_free_inference.ipynb)
![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)
![Model: Wan 2.1](https://img.shields.io/badge/Model-Wan2.1--1.3B-blue.svg)
![Hardware: Tesla T4](https://img.shields.io/badge/GPU-Tesla%20T4%20(15GB)-green.svg)

Generate high-quality videos using **Alibaba's Wan 2.1 (T2V-1.3B)** on a **100% free Google Colab instance** equipped with an NVIDIA Tesla T4 GPU (15 GB VRAM).

---

### 🔬 Rigorous Hardware & Complexity Breakdown

#### 1. Why Did Default Wan 2.1 Take >20 Minutes on Colab T4?
1. **Dynamic 4-bit Dequantization Penalty:** On Turing GPUs (`sm_75`, Tesla T4), `bitsandbytes` 4-bit (NF4) dequantizes weights dynamically to FP16 in CUDA register space for every matrix multiplication. On Turing, this runs **~1.8x slower per step** than native FP16.
2. **Quadratic Token Scaling $\mathcal{O}(S^2)$:** Wan 2.1 compresses space by $8\times$ and time by $4\times$.  
   $$\text{Total 3D Latent Tokens: } S = \left(\left\lfloor\frac{F-1}{4}\right\rfloor + 1\right) \times \left(\frac{H}{8}\right) \times \left(\frac{W}{8}\right)$$
   - At $832\times 480$ (33 frames): $S = 9 \times 60 \times 104 = \mathbf{56,160\text{ tokens}} \implies S^2 \approx \mathbf{3.15 \times 10^9}$.  
   - At $624\times 352$ (25 frames): $S = 7 \times 44 \times 78 = \mathbf{24,024\text{ tokens}} \implies S^2 \approx \mathbf{0.58 \times 10^9}$ (**5.4x less attention computation!**).  
   - At $512\times 320$ (17 frames): $S = 5 \times 40 \times 64 = \mathbf{12,800\text{ tokens}} \implies S^2 \approx \mathbf{0.16 \times 10^9}$ (**19.7x less attention computation!**).  
3. **Double Forward Pass with CFG:** Standard Classifier-Free Guidance (`guidance_scale=5.0`) executes **2 forward passes per step** (conditional + unconditional). 25 steps $\to$ 50 forward passes of the 1.3B DiT.
4. **Slow Weight Download:** 14 GB of weights without `hf-transfer` takes 5–8 minutes.

#### 2. The Speed Solution:
- **Native FP16:** The 1.3B DiT weighs only **2.6 GB** in FP16! With Model CPU offloading, peak VRAM during diffusion is only **~7.8 GB** (well within 15 GB). Native FP16 utilizes Turing Tensor Cores directly with zero dequantization overhead (**~1.8x faster**).
- **UniPC Multistep Scheduler:** Higher-order predictor-corrector solver achieving near-identical convergence in **16 steps** vs 25-30 steps with Euler (**~1.6x faster**).
- **Balanced Presets:** $624\times 352$ (25 frames) cuts quadratic attention tokens by **$5.4\times$**, delivering complete videos in **~2.8 - 3.5 minutes** instead of 25 minutes!
- **Fast HuggingFace Download:** `hf-transfer` cuts weight download to **~60 seconds**.

## 1. Verify GPU Runtime
Ensure you have selected **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi

## 2. Install Required Libraries & Enable Fast Downloads
Installs HuggingFace `diffusers`, `transformers`, `accelerate`, `hf-transfer`, and `gradio`.

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Fix Colab conflict: remove outdated torchao and install latest diffusers stack
!pip uninstall -y torchao
!pip install -q -U "diffusers>=0.33.0" "transformers>=4.48.0" "accelerate>=0.34.0" bitsandbytes gradio imageio[ffmpeg] sentencepiece protobuf hf-transfer


## 3. Load the Hardware-Optimized Wan 2.1 Pipeline
Loads the official `Wan-AI/Wan2.1-T2V-1.3B-Diffusers` model with:
- **Native FP16 Tensor Cores** (fastest compute path on Turing T4, 2.6 GB DiT footprint)
- **Model CPU Offloading** (4.8B UMT5 offloads to RAM after prompt encoding)
- **UniPC Multistep Scheduler** (optimal convergence in 14-18 steps)
- **3D VAE Tiling & Slicing** (memory bounded decoding)

In [ ]:
import os
import gc
import torch

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Prevent Colab torchao conflict
try:
    import diffusers.utils.import_utils as _iu
    _iu._torchao_available = False
except Exception:
    pass
try:
    import torchao.quantization
    if not hasattr(torchao.quantization, "FqnToConfig"):
        class FqnToConfig:
            def __init__(self, *args, **kwargs): pass
        torchao.quantization.FqnToConfig = FqnToConfig
except Exception:
    pass

# Configure CUDA optimizations for Turing T4 (sm_75)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.backends.cuda.enable_flash_sdp(False)        # Avoid unsupported warning on T4
        torch.backends.cuda.enable_mem_efficient_sdp(True) # Accelerated Cutlass kernel for Turing
        torch.backends.cuda.enable_math_sdp(True)
    except Exception:
        pass

from diffusers import AutoencoderKLWan, WanPipeline, WanTransformer3DModel, BitsAndBytesConfig
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

clear_gpu_memory()

# ── Precision & Quantization Mode ──────────────────────────────────────────
# "fp16" : Recommended on T4! Uses native Turing Tensor Cores (~1.8x faster, ~7.8GB peak VRAM).
# "4bit" : 0.5 bytes/param NF4 via bitsandbytes (~4.5GB VRAM, but ~1.8x slower on T4 due to dequant).
# "8bit" : 1.0 byte/param INT8 via bitsandbytes (~6.0GB VRAM).
QUANTIZATION_MODE = "fp16"

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
print(f"[1/3] Loading VAE (float32 with tiling) from {model_id}...")
vae = AutoencoderKLWan.from_pretrained(
    model_id,
    subfolder="vae",
    torch_dtype=torch.float32,
)

print(f"[2/3] Loading 1.3B DiT in {QUANTIZATION_MODE.upper()} mode...")
transformer = None
if QUANTIZATION_MODE == "4bit":
    print("  -> Applying 4-BIT NF4 quantization (0.5 bytes/param)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    transformer = WanTransformer3DModel.from_pretrained(
        model_id,
        subfolder="transformer",
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
    )
elif QUANTIZATION_MODE == "8bit":
    print("  -> Applying 8-BIT INT8 quantization (1.0 byte/param)...")
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    transformer = WanTransformer3DModel.from_pretrained(
        model_id,
        subfolder="transformer",
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
    )
else:
    print("  -> Native FP16 (2 bytes/param): 1.3B DiT = 2.6 GB. Runs directly on Tensor Cores.")

pipe_kwargs = {"vae": vae, "torch_dtype": torch.float16}
if transformer is not None:
    pipe_kwargs["transformer"] = transformer

print("  -> Assembling WanPipeline...")
pipe = WanPipeline.from_pretrained(model_id, **pipe_kwargs)

# Switch to UniPC solver (converges in 14-18 steps for 480p/360p)
try:
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
    print("  -> UniPC Multistep Scheduler activated (16 steps optimal).")
except Exception as e:
    print(f"  -> Scheduler note: {e}")

print("  -> Enabling Model CPU Offload (UMT5-XXL 4.8B offloads to RAM after prompt encoding)...")
pipe.enable_model_cpu_offload()

print("[3/3] Enabling 3D VAE Tiling & Slicing...")
if hasattr(pipe, "vae") and pipe.vae is not None:
    if hasattr(pipe.vae, "enable_tiling"):
        pipe.vae.enable_tiling()
    if hasattr(pipe.vae, "enable_slicing"):
        pipe.vae.enable_slicing()

allocated = torch.cuda.memory_allocated() / (1024**3)
print(f"✓ Wan 2.1 Pipeline Ready ({QUANTIZATION_MODE.upper()})! GPU Memory: {allocated:.2f} GB")


[1/3] Loading VAE from Wan-AI/Wan2.1-T2V-1.3B-Diffusers...
[2/3] Loading 1.3B DiT in 4BIT mode...
  -> Applying 4-BIT NF4 quantization (0.5 bytes/parameter)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

## 4. Quick Video Generation (Balanced Preset ~2.8 min)
Uses the hardware-optimized **Balanced Preset** (`624x352`, 25 frames, 16 steps):
- 3D Latent shape: `[7 x 44 x 78]` → **24,024 tokens** (vs 56,160 in 480p high)
- Denoising passes: 16 steps × 2 (CFG) = 32 forward passes
- **Execution Time:** ~2.5 - 3.5 minutes on Tesla T4 (instead of 25+ minutes!).

In [ ]:
from IPython.display import Video, display

prompt = "A sleek cyberpunk racing car drifting around a sharp mountain hairpin turn, tire smoke, reflections in wet asphalt, sunset lighting, cinematic 4k."
negative_prompt = "Bright, overexposed, static, low quality, distorted, blurry, deformed, flickering"

width = 624
height = 352
num_frames = 25       # 25 frames ≈ 1.5s at 16fps
steps = 16            # 16 steps with UniPC
guidance_scale = 5.0  # Classifier-Free Guidance
seed = 42

lt = (num_frames - 1) // 4 + 1
lh = height // 8
lw = width // 8
tokens = lt * lh * lw
passes = steps * 2

print(f"⚡ Starting Wan 2.1 Generation:")
print(f"   Resolution: {width}x{height} | Frames: {num_frames} ({num_frames/16:.1f}s)")
print(f"   3D Latent Shape: [{lt}x{lh}x{lw}] -> Tokens: {tokens:,}")
print(f"   Steps: {steps} (UniPC) | Forward Passes: {passes}")
print(f"   Estimated T4 Runtime: ~2.8 minutes")

clear_gpu_memory()

with torch.inference_mode():
    output = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=height,
        width=width,
        num_frames=num_frames,
        num_inference_steps=steps,
        guidance_scale=guidance_scale,
        generator=torch.Generator(device="cpu").manual_seed(seed),
    )

frames = output.frames[0] if hasattr(output, "frames") else output.images[0]
output_file = "demo_output.mp4"
export_to_video(frames, output_file, fps=16)
clear_gpu_memory()

print(f"✓ Video successfully generated: {output_file}")
display(Video(output_file, embed=True, width=640))


## 5. Launch Interactive Gradio Web App with Public Link
Run this cell to launch the web interface. Gradio will provide a public `https://xxxx.gradio.live` link so anyone can generate videos directly from their browser or phone!

In [ ]:
import random
import gradio as gr

RESOLUTION_MAP = {
    "624x352 (Widescreen 16:9 - Balanced, ~3 min)": (624, 352),
    "512x320 (Widescreen 16:9 - Fast Draft, ~1.5 min)": (512, 320),
    "832x480 (Cinematic 16:9 - High Quality, ~8-14 min)": (832, 480),
    "320x512 (Portrait 9:16 - Fast Draft, ~1.5 min)": (320, 512),
    "480x832 (Portrait 9:16 - High Quality, ~8-14 min)": (480, 832),
    "512x512 (Square 1:1 - Fast, ~2.5 min)": (512, 512),
    "624x624 (Square 1:1 - Standard, ~6 min)": (624, 624),
}

def estimate_metrics(resolution, num_frames, steps, guidance):
    w, h = RESOLUTION_MAP.get(resolution, (624, 352))
    lt = (int(num_frames) - 1) // 4 + 1
    lh = h // 8
    lw = w // 8
    tokens = lt * lh * lw
    passes = int(steps) * (2 if float(guidance) > 1.0 else 1)
    norm_s = tokens / 24000.0
    sec_per_pass = 1.35 * norm_s + 1.15 * (norm_s ** 2)
    est_sec = passes * sec_per_pass + 0.35 * int(num_frames) + 2.0
    est_min = est_sec / 60.0
    return (
        f"⏱️ **Estimated T4 Runtime:** `~{est_min:.1f} min` ({est_sec:.0f}s) | "
        f"🧩 **Tokens:** `{tokens:,}` latents `[{lt}x{lh}x{lw}]` | "
        f"🔁 **Passes:** `{passes}` ({steps} st {'x 2 (CFG)' if passes > steps else ''})"
    )

def set_preset(preset):
    if "Fast Draft" in preset:
        return "512x320 (Widescreen 16:9 - Fast Draft, ~1.5 min)", 17, 15, 5.0
    elif "Balanced" in preset:
        return "624x352 (Widescreen 16:9 - Balanced, ~3 min)", 25, 16, 5.0
    elif "Cinematic" in preset or "High" in preset:
        return "832x480 (Cinematic 16:9 - High Quality, ~8-14 min)", 33, 20, 5.0
    return "624x352 (Widescreen 16:9 - Balanced, ~3 min)", 25, 16, 5.0

def gradio_generate(prompt, neg_prompt, resolution, num_frames, steps, guidance_scale, seed, progress=gr.Progress(track_tqdm=True)):
    if not prompt or not prompt.strip():
        raise gr.Error("Please enter a prompt.")

    w, h = RESOLUTION_MAP.get(resolution, (624, 352))
    actual_seed = random.randint(0, 2**31 - 1) if seed == -1 else int(seed)

    clear_gpu_memory()
    out_path = f"wan_{actual_seed}.mp4"

    try:
        progress(0.05, desc="Encoding prompt text...")
        with torch.inference_mode():
            out = pipe(
                prompt=prompt,
                negative_prompt=neg_prompt,
                height=h,
                width=w,
                num_frames=int(num_frames),
                num_inference_steps=int(steps),
                guidance_scale=float(guidance_scale),
                generator=torch.Generator(device="cpu").manual_seed(actual_seed),
            )
        frames = out.frames[0] if hasattr(out, "frames") else out.images[0]
        export_to_video(frames, out_path, fps=16)
        clear_gpu_memory()
        return out_path, f"✓ Generated in {w}x{h} ({num_frames} frames)! Seed: {actual_seed}"
    except Exception as e:
        clear_gpu_memory()
        return None, f"Error: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), title="Wan 2.1 Colab T4") as demo:
    gr.Markdown("# 🎬 Wan 2.1 Video Generator (Free Colab T4 16GB)")
    gr.Markdown("Powered by **Alibaba Wan 2.1 (T2V-1.3B)**. Accelerated via native FP16 Tensor Cores, UniPC scheduler, and Model CPU offload under 8 GB VRAM.")

    with gr.Row():
        with gr.Column(scale=5):
            p_in = gr.Textbox(label="Prompt", lines=3, value="A sleek sports car drifting around a sharp mountain hairpin turn, tire smoke, sunset lighting, high speed camera.")
            with gr.Accordion("Negative Prompt", open=False):
                np_in = gr.Textbox(label="Negative Prompt", lines=2, value="Bright, overexposed, static, low quality, distorted, blurry, deformed")

            preset_in = gr.Dropdown(
                label="⚡ Speed / Quality Preset",
                choices=[
                    "⚡ Fast Draft (~1.5 min) [512x320, 17f, 15st]",
                    "⚡ Balanced [Recommended] (~3 min) [624x352, 25f, 16st]",
                    "🎬 Cinematic Standard (~8 min) [832x480, 25f, 18st]",
                    "💎 High Quality (~14 min) [832x480, 33f, 20st]",
                ],
                value="⚡ Balanced [Recommended] (~3 min) [624x352, 25f, 16st]",
            )

            with gr.Row():
                res_in = gr.Dropdown(label="Resolution", choices=list(RESOLUTION_MAP.keys()), value="624x352 (Widescreen 16:9 - Balanced, ~3 min)")
                f_in = gr.Slider(label="Frames", minimum=17, maximum=49, step=8, value=25)
            with gr.Row():
                s_in = gr.Slider(label="Steps", minimum=12, maximum=30, step=1, value=16)
                g_in = gr.Slider(label="Guidance Scale (CFG)", minimum=1.0, maximum=8.0, step=0.5, value=5.0)

            metrics_box = gr.Markdown(value=estimate_metrics("624x352 (Widescreen 16:9 - Balanced, ~3 min)", 25, 16, 5.0))

            with gr.Row():
                seed_in = gr.Number(label="Seed (-1 for random)", value=-1, precision=0)
                rnd_btn = gr.Button("🎲 Random", scale=1)
            rnd_btn.click(lambda: -1, outputs=[seed_in])
            gen_btn = gr.Button("🚀 Generate Video", variant="primary", size="lg")

        with gr.Column(scale=5):
            vid_out = gr.Video(label="Generated Video", autoplay=True, loop=True)
            status_out = gr.Markdown("Ready to generate.")

    preset_in.change(fn=set_preset, inputs=[preset_in], outputs=[res_in, f_in, s_in, g_in]).then(
        fn=estimate_metrics, inputs=[res_in, f_in, s_in, g_in], outputs=[metrics_box]
    )
    for comp in [res_in, f_in, s_in, g_in]:
        comp.change(fn=estimate_metrics, inputs=[res_in, f_in, s_in, g_in], outputs=[metrics_box])

    gen_btn.click(fn=gradio_generate, inputs=[p_in, np_in, res_in, f_in, s_in, g_in, seed_in], outputs=[vid_out, status_out])

# Launch public link with share=True
demo.queue().launch(share=True, debug=True)
